# Aral Sea -- Early 2000s vs 2025

Side-by-side comparison of two median composites:
- **Landsat 5/7** (summer 2000-2003, 30 m) -- the Aral Sea was still partially intact
- **Sentinel-2** (summer 2025, 10 m) -- near-complete desiccation of the South Aral

**Workflow:**
1. Search Planetary Computer STAC for the least-cloudy summer scenes
2. Download each band clipped to the AOI and save as local GeoTIFF
3. Build median mosaics from the local files (fast, no re-download on re-run)
4. Visualise true colour and NDWI side by side


## 1. Imports and configuration


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_bounds as make_transform
from rasterio.crs import CRS

import pystac_client
import planetary_computer

# ---------------------------------------------------------------------------
# Aral Sea AOI (WGS84: lon_min, lat_min, lon_max, lat_max)
BBOX = [57.5, 43.5, 62.5, 46.8]

# Summer window -- clear skies over Central Asia
SEASON = ('07-01', '09-30')

# Landsat early-2000s window and Sentinel-2 year
LS_YEARS  = range(2000, 2004)   # pull best scenes across 2000-2003
S2_YEAR   = 2025

MAX_SCENES = 10    # scenes per sensor to download and stack
RESOLUTION = 300   # metres -- change to 100 for sharper output (slower)

# Output CRS: UTM 41N covers the Aral Sea
TARGET_CRS     = CRS.from_epsg(32641)
WGS84          = CRS.from_epsg(4326)
UTM_BOUNDS     = transform_bounds(WGS84, TARGET_CRS, *BBOX)
GRID_WIDTH     = int((UTM_BOUNDS[2] - UTM_BOUNDS[0]) / RESOLUTION)
GRID_HEIGHT    = int((UTM_BOUNDS[3] - UTM_BOUNDS[1]) / RESOLUTION)
GRID_TRANSFORM = make_transform(*UTM_BOUNDS, GRID_WIDTH, GRID_HEIGHT)

# Local download directories
DIR_LS = 'data/downloads/landsat_early'
DIR_S2 = 'data/downloads/s2_2025'
os.makedirs(DIR_LS, exist_ok=True)
os.makedirs(DIR_S2, exist_ok=True)

print(f'Grid : {GRID_WIDTH} x {GRID_HEIGHT} px  @ {RESOLUTION} m/px')
print(f'Download dirs: {DIR_LS}  |  {DIR_S2}')


## 2. Connect to Planetary Computer


In [ ]:
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
print('Connected:', catalog.title)


## 3. Band helpers

`download_band()` reads only the COG tiles overlapping the AOI
(rasterio does a windowed read internally) and saves the result as a
local GeoTIFF.  On subsequent runs the file is reused without re-downloading.


In [ ]:
BAND_ALIASES = {
    'blue':   ['blue',  'B02'],
    'green':  ['green', 'B03'],
    'red':    ['red',   'B04'],
    'nir08':  ['nir08', 'nir', 'B08', 'B8A'],
    'swir16': ['swir16', 'swir', 'B11'],
}


def get_asset_href(item, common_name):
    aliases = BAND_ALIASES.get(common_name, [common_name])
    for asset in item.assets.values():
        for band in asset.extra_fields.get('eo:bands', []):
            if band.get('common_name') in aliases:
                return asset.href
    for alias in aliases:
        if alias in item.assets:
            return item.assets[alias].href
    raise KeyError(f"Band '{common_name}' not found. Available: {list(item.assets.keys())}")


def reproject_to_grid(href):
    """Read a COG band clipped to the AOI and reproject to the common UTM grid."""
    out = np.zeros((GRID_HEIGHT, GRID_WIDTH), dtype=np.float32)
    with rasterio.open(href) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=GRID_TRANSFORM,
            dst_crs=TARGET_CRS,
            resampling=Resampling.average,
            src_nodata=src.nodata,
            dst_nodata=0.0,
        )
    return out


def download_band(item, band_name, save_dir, scale, offset=0.0):
    """
    Download one band from a STAC item, apply scale/offset, and save
    as a local float32 GeoTIFF aligned to the common UTM grid.
    Returns the local file path.
    Skips the download if the file already exists.
    """
    fname = os.path.join(save_dir, f'{item.id}__{band_name}.tif')
    if os.path.exists(fname):
        return fname

    href = get_asset_href(item, band_name)
    arr  = reproject_to_grid(href)
    arr  = np.clip(arr.astype(np.float32) * scale + offset, 0, 1)

    with rasterio.open(
        fname, 'w', driver='GTiff',
        height=GRID_HEIGHT, width=GRID_WIDTH,
        count=1, dtype='float32',
        crs=TARGET_CRS, transform=GRID_TRANSFORM,
        nodata=0.0,
    ) as dst:
        dst.write(arr, 1)

    return fname


def median_mosaic_from_files(file_paths):
    """
    Load a set of local GeoTIFFs and return their nanmedian as a 2-D array.
    Fill pixels (value == 0) are treated as NaN before the median.
    """
    stack = []
    for fp in file_paths:
        with rasterio.open(fp) as src:
            arr = src.read(1).astype(np.float32)
        arr[arr <= 0] = np.nan
        stack.append(arr)
    return np.nan_to_num(
        np.nanmedian(np.stack(stack, axis=0), axis=0),
        nan=0.0,
    ).astype(np.float32)


print('Helpers ready.')


## 4. Landsat search and download (2000-2003)

We collect the least-cloudy summer scenes across the 2000-2003 window.
Taking scenes from multiple years gives better spatial coverage.


In [ ]:
ls_all = []
for year in LS_YEARS:
    search = catalog.search(
        collections=['landsat-c2-l2'],
        bbox=BBOX,
        datetime=f'{year}-{SEASON[0]}/{year}-{SEASON[1]}',
        query={'eo:cloud_cover': {'lt': 30}},
    )
    ls_all.extend(search.items())

ls_all.sort(key=lambda x: x.properties.get('eo:cloud_cover', 100))
ls_selected = ls_all[:MAX_SCENES]

print(f'Found {len(ls_all)} Landsat scenes (2000-2003), using top {len(ls_selected)}')
for item in ls_selected:
    date  = item.properties.get('datetime', '')[:10]
    cloud = item.properties.get('eo:cloud_cover', float('nan'))
    plat  = item.properties.get('platform', '?')
    print(f'  {date}  {plat:<12}  cloud={cloud:.1f}%')


In [ ]:
LS_SCALE, LS_OFFSET = 0.0000275, -0.2
LS_BANDS = ('blue', 'green', 'red', 'nir08', 'swir16')

ls_files = {b: [] for b in LS_BANDS}

for i, item in enumerate(ls_selected):
    print(f'[{i+1}/{len(ls_selected)}] {item.id[:40]} ...', end=' ', flush=True)
    for band in LS_BANDS:
        try:
            fp = download_band(item, band, DIR_LS, scale=LS_SCALE, offset=LS_OFFSET)
            ls_files[band].append(fp)
        except Exception as e:
            print(f'\n  WARNING {band}: {e}', end='')
    print('OK')

print(f'\nLandsat files saved to: {DIR_LS}')


## 5. Sentinel-2 search and download (2025)


In [ ]:
s2_search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime=f'{S2_YEAR}-{SEASON[0]}/{S2_YEAR}-{SEASON[1]}',
    query={'eo:cloud_cover': {'lt': 20}},
)
s2_all = sorted(s2_search.items(),
                key=lambda x: x.properties.get('eo:cloud_cover', 100))
s2_selected = s2_all[:MAX_SCENES]

print(f'Found {len(s2_all)} Sentinel-2 scenes ({S2_YEAR}), using top {len(s2_selected)}')
for item in s2_selected:
    date  = item.properties.get('datetime', '')[:10]
    cloud = item.properties.get('eo:cloud_cover', float('nan'))
    print(f'  {date}  cloud={cloud:.1f}%')


In [ ]:
S2_BANDS = ('blue', 'green', 'red', 'nir08', 'swir16')

s2_files = {b: [] for b in S2_BANDS}

for i, item in enumerate(s2_selected):
    print(f'[{i+1}/{len(s2_selected)}] {item.id[:40]} ...', end=' ', flush=True)
    for band in S2_BANDS:
        try:
            fp = download_band(item, band, DIR_S2, scale=0.0001)
            s2_files[band].append(fp)
        except Exception as e:
            print(f'\n  WARNING {band}: {e}', end='')
    print('OK')

print(f'\nSentinel-2 files saved to: {DIR_S2}')


## 6. Build median mosaics from local files

All computation from here reads from disk -- no network calls.
Re-running this cell is instant if the files are already downloaded.


In [ ]:
print('Building Landsat mosaic ...', end=' ', flush=True)
ls_mosaic = {b: median_mosaic_from_files(ls_files[b]) for b in LS_BANDS}
coverage = np.mean(ls_mosaic['green'] > 0) * 100
print(f'coverage = {coverage:.0f}%')

print('Building Sentinel-2 mosaic ...', end=' ', flush=True)
s2_mosaic = {b: median_mosaic_from_files(s2_files[b]) for b in S2_BANDS}
coverage = np.mean(s2_mosaic['green'] > 0) * 100
print(f'coverage = {coverage:.0f}%')


## 7. True colour comparison


In [ ]:
def to_rgb(mosaic):
    """Stack R/G/B bands and apply per-channel 2-98% stretch."""
    rgb = np.stack([mosaic['red'], mosaic['green'], mosaic['blue']], axis=-1)
    for ch in range(3):
        band = rgb[..., ch]
        valid = band[band > 0]
        if valid.size:
            lo, hi = np.percentile(valid, [2, 98])
            rgb[..., ch] = np.clip((band - lo) / (hi - lo + 1e-9), 0, 1)
    return rgb


ls_rgb = to_rgb(ls_mosaic)
s2_rgb = to_rgb(s2_mosaic)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), subplot_kw={'aspect': 'equal'})
fig.suptitle('Aral Sea -- True colour median composite', fontsize=14)

axes[0].imshow(ls_rgb, interpolation='bilinear')
axes[0].set_title('Landsat 5/7  (30 m)\nEarly 2000s (2000-2003 summer median)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(s2_rgb, interpolation='bilinear')
axes[1].set_title(f'Sentinel-2  (10 m)\n{S2_YEAR} summer median', fontsize=11)
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 8. NDWI comparison


In [ ]:
def compute_ndwi(mosaic):
    g, n = mosaic['green'], mosaic['nir08']
    ndwi = (g - n) / (g + n + 1e-9)
    ndwi[(g == 0) & (n == 0)] = np.nan
    return ndwi


ls_ndwi = compute_ndwi(ls_mosaic)
s2_ndwi = compute_ndwi(s2_mosaic)

# Shared colour scale from actual data
combined = np.concatenate([
    ls_ndwi[np.isfinite(ls_ndwi)],
    s2_ndwi[np.isfinite(s2_ndwi)],
])
vmin, vmax = np.percentile(combined, [2, 98])

fig, axes = plt.subplots(1, 2, figsize=(18, 8), subplot_kw={'aspect': 'equal'})
fig.suptitle('Aral Sea -- NDWI  (blue = water)', fontsize=14)

for ax, ndwi, title in [
    (axes[0], ls_ndwi, 'Landsat 5/7  (30 m) -- Early 2000s'),
    (axes[1], s2_ndwi, f'Sentinel-2  (10 m) -- {S2_YEAR}'),
]:
    im = ax.imshow(ndwi, cmap='RdYlBu', vmin=vmin, vmax=vmax, interpolation='bilinear')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='NDWI')

plt.tight_layout()
plt.show()

# Water area estimate (NDWI > 0)
pix_km2 = (RESOLUTION / 1000) ** 2
for label, ndwi in [('Early 2000s (Landsat)', ls_ndwi), (f'{S2_YEAR} (Sentinel-2)', s2_ndwi)]:
    area = int(np.nansum(ndwi > 0)) * pix_km2
    print(f'{label}: ~{area:,.0f} km2 of open water (NDWI > 0)')
